# TaxSetu AI: bank-narration classifier
Fine-tunes a small BERT (`prajjwal1/bert-mini`, ~11M parameters) to file a bank-statement row under a ledger head, exports it as an int8 ONNX model, and re-tests the exported file the way the API will run it.

**Data:** synthetic (`narration_data.py`); no customer data. The test set uses phrases held out of training. Retrain with real, labelled narrations before relying on it.

Runtime > Change runtime type > T4 GPU, then Runtime > Run all.

In [ ]:
!pip -q install onnx onnxruntime tokenizers

In [ ]:
%%writefile narration_data.py
"""Synthetic training data for the bank-narration classifier (no real customer data is used or stored).

A bank-statement row such as "UPI/DR/412345678901/BESCOM/okhdfcbank/Bill Pay" is labelled with the ledger head a shop owner would file it under.
Every label has a pool of phrases and merchants. The LAST fifth of each pool is held back for the test set only, so the test measures
whether the model handles phrases it never saw in training (typos, case, bank formats and reference numbers are randomised for both).
Pure standard library so the same file runs inside the Colab notebook.
"""
import random

LABELS = ["Bank Charges", "Electricity Charges", "Insurance", "Interest Paid", "Interest Received", "Office Expenses", "Other", "Printing and Stationery",
          "Professional Fees", "Purchases", "Rent Paid", "Repairs and Maintenance", "Salary and Wages", "Sales", "Telephone and Internet", "Travelling Expenses"]
INCOMING = {"Sales", "Interest Received"}          # the other labels are normally money going out; "Other" occurs in both directions

POOL = {
    "Bank Charges": ["SMS CHARGES", "SMS ALERT CHGS", "ANNUAL FEE", "CHQ BOOK ISSUE CHGS", "MIN BAL PENALTY", "GST ON CHARGES", "DEBIT CARD AMC", "BANK CHARGES",
                     "NEFT CHARGES", "IMPS CHG", "ATM ANNUAL FEE", "ACCOUNT MAINTENANCE CHG", "CASH HANDLING CHARGES", "RTGS CHARGES"],
    "Electricity Charges": ["BESCOM", "MSEDCL", "TATA POWER", "BSES RAJDHANI", "TNEB", "KSEB", "TORRENT POWER", "ELECTRICITY BILL", "BIJLI BILL", "POWER BILL",
                            "CESC", "ADANI ELECTRICITY", "UPPCL", "JBVNL", "BESL", "HESCOM", "DHBVN", "NPCL"],
    "Insurance": ["LIC PREMIUM", "INSURANCE PREMIUM", "HDFC LIFE", "ICICI LOMBARD", "STAR HEALTH", "NEW INDIA ASSURANCE", "VEHICLE INSURANCE", "FIRE POLICY",
                  "BAJAJ ALLIANZ", "POLICY RENEWAL", "MAX LIFE", "ORIENTAL INSURANCE", "SHOP INSURANCE", "SBI GENERAL"],
    "Interest Paid": ["INT PD", "LOAN INTEREST", "INTEREST DEBITED", "OD INTEREST", "CC INTEREST", "EMI INTEREST PART", "INTEREST ON LOAN", "INT.PD",
                      "INTEREST CHARGED", "TERM LOAN INTEREST", "INT ON CASH CREDIT", "INTEREST DEBIT"],
    "Interest Received": ["INT CR", "INTEREST CREDITED", "SAVINGS INTEREST", "FD INTEREST", "INT.PAID TILL", "INTEREST RECEIVED", "SB INT", "QUARTERLY INTEREST CREDIT",
                          "INT ON FD", "INTEREST PAYOUT", "RD INTEREST", "CREDIT INTEREST"],
    "Office Expenses": ["OFFICE TEA", "PANTRY ITEMS", "OFFICE CLEANING", "HOUSEKEEPING", "OFFICE SUPPLIES", "WATER CAN", "OFFICE EXPENSES", "CHAI AND SNACKS",
                        "OFFICE MISC", "DRINKING WATER", "OFFICE REFRESHMENTS", "SWEEPER CHARGES"],
    "Printing and Stationery": ["STATIONERY", "XEROX", "PHOTOCOPY", "PRINTING CHARGES", "PAPER REAMS", "PEN AND FILES", "OFFICE STATIONERY", "VISITING CARDS PRINT",
                                "FLEX PRINTING", "KOKUYO", "CAMLIN", "BILL BOOK PRINTING", "LETTERHEAD PRINTING", "CLASSMATE NOTEBOOKS"],
    "Professional Fees": ["CA FEES", "AUDIT FEES", "LEGAL FEES", "ADVOCATE FEES", "CONSULTANT FEES", "PROFESSIONAL CHARGES", "GST RETURN FILING FEES", "TAX CONSULTANT",
                          "COMPANY SECRETARY FEES", "ROC FILING CHARGES", "ACCOUNTANT FEES", "BOOK KEEPING CHARGES", "TDS RETURN FEES", "ITR FILING FEES"],
    "Purchases": ["PURCHASE OF GOODS", "STOCK PURCHASE", "RAW MATERIAL", "SUPPLIER INVOICE", "WHOLESALE PURCHASE", "MATERIAL PURCHASE", "GOODS PAYMENT", "MERCHANDISE",
                  "SHREE TRADERS", "GUPTA AGENCIES", "BALAJI DISTRIBUTORS", "SAI ENTERPRISES", "MAHALAXMI WHOLESALE", "KRISHNA SUPPLIERS", "METRO CASH AND CARRY",
                  "UDAAN", "JUMBOTAIL", "OM TRADING CO", "SURESH WHOLESALERS", "STOCK PAYMENT"],
    "Rent Paid": ["SHOP RENT", "OFFICE RENT", "HOUSE RENT", "RENT FOR APRIL", "GODOWN RENT", "KIRAYA", "MONTHLY RENT", "RENT PAYMENT", "LEASE RENT", "WAREHOUSE RENT",
                  "RENT TO LANDLORD", "PREMISES RENT", "RENT FOR SHOP", "RENT MAY"],
    "Repairs and Maintenance": ["REPAIR", "AC REPAIR", "PLUMBER", "ELECTRICIAN", "MAINTENANCE CHARGES", "AMC SERVICE", "COMPUTER REPAIR", "MISTRI", "PAINTING WORK",
                                "LIFT MAINTENANCE", "SERVICING", "SHUTTER REPAIR", "REPAIRS AND MAINTENANCE", "CARPENTER CHARGES"],
    "Salary and Wages": ["SALARY", "WAGES", "SALARY FOR MARCH", "STAFF SALARY", "PAYROLL", "LABOUR WAGES", "MAZDOORI", "DAILY WAGES", "SALARY ADVANCE", "SALARIES",
                         "STAFF PAYMENT", "WORKER WAGES", "MONTHLY SALARY", "LOADING UNLOADING LABOUR"],
    "Sales": ["SALES", "CUSTOMER PAYMENT", "POS SETTLEMENT", "RAZORPAY SETTLEMENT", "PAYTM SETTLEMENT", "PHONEPE SETTLEMENT", "INVOICE RECEIPT", "BILL PAYMENT RECEIVED",
              "SALES COLLECTION", "UPI COLLECTION", "CARD SETTLEMENT", "CASHFREE SETTLEMENT", "BHARATPE SETTLEMENT", "SALE PROCEEDS", "PAYMENT AGAINST INVOICE", "QR COLLECTION"],
    "Telephone and Internet": ["AIRTEL", "JIO", "BSNL", "VODAFONE IDEA", "VI POSTPAID", "ACT FIBERNET", "HATHWAY BROADBAND", "MOBILE RECHARGE", "BROADBAND BILL",
                               "TELEPHONE BILL", "INTERNET CHARGES", "TATA PLAY FIBER", "JIOFIBER", "LANDLINE BILL", "WIFI BILL", "DATA RECHARGE"],
    "Travelling Expenses": ["PETROL", "DIESEL", "HPCL", "IOCL", "BPCL", "SHELL FUEL", "OLA CABS", "UBER", "IRCTC", "REDBUS", "FASTAG RECHARGE", "TOLL", "TRAVEL",
                            "AUTO FARE", "FLIGHT TICKET", "INDIGO", "TRAIN TICKET", "FUEL PUMP", "TAXI FARE", "BUS TICKET"],
    "Other": ["TRANSFER TO SELF", "ATM WDL", "CASH WITHDRAWAL", "FD BOOKING", "MF SIP", "FAMILY TRANSFER", "ZOMATO", "SWIGGY", "AMAZON PAY", "PERSONAL", "GIFT",
              "LOAN FROM FRIEND", "CHEQUE DEPOSIT", "UNKNOWN", "SELF", "MYNTRA", "BOOKMYSHOW", "SCHOOL FEES", "GROCERIES HOME", "DONATION TEMPLE", "MEDICAL STORE"],
}
PEOPLE = ["RAMESH KUMAR", "S REDDY", "MRS IYER", "P SHARMA", "M/S GUPTA PROPERTIES", "R PATEL", "ANITA SINGH", "MOHAN LAL", "K VENKATESH", "SUNITA DEVI",
          "ABDUL RAHIM", "J D'SOUZA", "VIKRAM JOSHI", "LAKSHMI NAIR", "BHARAT TRADERS", "NEHA VERMA", "GURPREET SINGH", "T MURUGAN"]
PERSON_LABELS = {"Rent Paid", "Salary and Wages", "Repairs and Maintenance", "Professional Fees", "Purchases", "Sales", "Other"}
NOTES = ["", "", "", "BILL", "PAYMENT", "MAR 2026", "APR 26", "INV 4821", "REF", "ONLINE", "THANKS", "FEB", "Q3", "NO 12", "PAID", "HELP"]
BANKS = ["okhdfcbank", "oksbi", "okicici", "ybl", "paytm", "axl", "ibl", "upi", "apl"]
IFSC = ["HDFC0001234", "SBIN0004567", "ICIC0002345", "CNRB0001122", "UTIB0000987", "PUNB0123400", "KKBK0000571", "BARB0KORAMA"]
OUT_FORMATS = ["UPI/DR/{ref}/{m}/{vpa}/{note}", "NEFT-{ifsc}-{m}-{note}", "IMPS/P2A/{ref}/{m}", "POS {card} {m}", "ACH D- {m}-{ref}", "{m}", "{note} {m}", "BILLPAY {m} {ref}",
               "CHQ PAID {ref} {m}", "{m} {note}", "PAID TO {m} FOR {note}", "{note} - {m}", "ONLINE PMT {m}", "UPI-{m}-{vpa}-{ref}-{note}", "DEBIT {m} {ref}", "BY CASH {m}"]
IN_FORMATS = ["UPI/CR/{ref}/{m}/{vpa}", "NEFT CR-{ifsc}-{m}-{note}", "IMPS/{ref}/{m}", "BY TRANSFER-{m}", "CREDIT {m} {ref}", "{m}", "{note} {m}", "{m} {note}",
              "UPI-{m}-{vpa}-{ref}", "RTGS CR {ifsc} {m}", "INW {m} {ref}"]


def split_pool(label):
    pool = POOL[label]
    cut = len(pool) - max(2, len(pool) // 5)
    return pool[:cut], pool[cut:]


def _noise(rng, s):
    s = rng.choice((s, s, s.lower(), s.title()))
    if len(s) > 4 and rng.random() < 0.12:               # a typo: drop or swap a letter
        i = rng.randrange(1, len(s) - 1)
        s = s[:i] + s[i + 1:] if rng.random() < 0.5 else s[:i] + s[i + 1] + s[i] + s[i + 2:]
    return s


def _one(rng, label, part):
    train, held = split_pool(label)
    phrase = rng.choice(train if part == "train" else held)
    if label in PERSON_LABELS and rng.random() < 0.35:
        who = rng.choice(PEOPLE)
        phrase = f"{who} {phrase}" if rng.random() < 0.5 else f"{phrase} {who}"
    incoming = label in INCOMING or (label == "Other" and rng.random() < 0.4)
    ref = str(rng.randrange(10 ** 8, 10 ** 12))
    text = rng.choice(IN_FORMATS if incoming else OUT_FORMATS).format(
        m=phrase, ref=ref, vpa=f"{rng.choice(PEOPLE).split()[0].lower()}@{rng.choice(BANKS)}", note=rng.choice(NOTES), ifsc=rng.choice(IFSC),
        card=f"{rng.randrange(4000, 6000)}XXXXXXXX{rng.randrange(1000, 9999)}")
    text = " ".join(_noise(rng, text).split())
    return {"text": ("in | " if incoming else "out | ") + text, "label": label}


def make(seed=7, n_train=40000, n_test=4000):
    """(train, test): lists of {"text", "label"}. Test rows use only the held-back phrases of each label."""
    rng = random.Random(seed)
    train = [_one(rng, LABELS[i % len(LABELS)], "train") for i in range(n_train)]
    test = [_one(rng, LABELS[i % len(LABELS)], "test") for i in range(n_test)]
    rng.shuffle(train)
    return train, test


if __name__ == "__main__":
    a, b = make(1, 6, 3)
    for r in a + b:
        print(r["label"].ljust(26), r["text"])

In [ ]:
import json, time, torch, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, f1_score
from narration_data import make, LABELS

SEED, BASE, EPOCHS, BATCH, LR, MAXLEN = 7, "prajjwal1/bert-mini", 3, 64, 3e-4, 64
torch.manual_seed(SEED)
train, test = make(seed=SEED, n_train=40000, n_test=4000)
val, train = train[:2000], train[2000:]            # val: same phrases as training (a sanity check); test: phrases never seen in training
label2id = {l: i for i, l in enumerate(LABELS)}
tok = AutoTokenizer.from_pretrained("bert-base-uncased")      # prajjwal1/bert-mini ships no tokenizer files; it uses the bert-base-uncased vocabulary
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev, "| train", len(train), "val", len(val), "test", len(test))
from transformers import BertConfig, BertForSequenceClassification
cfg = BertConfig(vocab_size=30522, hidden_size=256, num_hidden_layers=4, num_attention_heads=4, intermediate_size=1024, num_labels=len(LABELS),
                 id2label=dict(enumerate(LABELS)), label2id=label2id)      # bert-mini's shape; its repo's config.json has no model_type, so the Auto classes cannot read it
try:
    model, PRETRAINED = BertForSequenceClassification.from_pretrained(BASE, config=cfg), True
except Exception as exc:
    print("could not load the pretrained weights; training from random initialisation:", repr(exc)[:300])
    model, PRETRAINED = BertForSequenceClassification(cfg), False
model = model.to(dev)
print("pretrained weights loaded:", PRETRAINED)

def encode(rows):
    e = tok([r["text"] for r in rows], truncation=True, max_length=MAXLEN, padding=True, return_tensors="pt")
    return {k: v.to(dev) for k, v in e.items()}, torch.tensor([label2id[r["label"]] for r in rows]).to(dev)

def predict(rows):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(rows), 256):
            x, _ = encode(rows[i:i + 256]); out += model(**x).logits.argmax(-1).tolist()
    return out

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = EPOCHS * ((len(train) + BATCH - 1) // BATCH)
sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
t0 = time.time()
for epoch in range(EPOCHS):
    model.train(); order = np.random.RandomState(SEED + epoch).permutation(len(train)); total = 0.0
    for i in range(0, len(train), BATCH):
        x, y = encode([train[j] for j in order[i:i + BATCH]])
        loss = torch.nn.functional.cross_entropy(model(**x).logits, y)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); opt.zero_grad(); total += loss.item()
        if (i // BATCH) % 100 == 0: print(f"  epoch {epoch + 1} step {i // BATCH}/{len(train) // BATCH}  loss {loss.item():.4f}  ({time.time() - t0:.0f}s)", flush=True)
    acc = np.mean([LABELS[p] == r["label"] for p, r in zip(predict(val), val)])
    print(f"epoch {epoch + 1}/{EPOCHS}  loss {total / (len(train) / BATCH):.4f}  val accuracy {acc:.4f}  ({time.time() - t0:.0f}s)")

pred = [LABELS[p] for p in predict(test)]
gold = [r["label"] for r in test]
fp32_acc = float(np.mean([p == g for p, g in zip(pred, gold)]))
print("\nHELD-OUT test accuracy (fp32):", round(fp32_acc, 4), "| macro-F1:", round(f1_score(gold, pred, average="macro"), 4))
print(classification_report(gold, pred, digits=3))

In [ ]:
import os, zipfile, onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
from tokenizers import Tokenizer

OUT = "taxsetu_narration_model"; os.makedirs(OUT, exist_ok=True)
model.eval().cpu()
class Wrap(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, input_ids, attention_mask, token_type_ids): return self.m(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids).logits
e = tok(["out | UPI/DR/1234/BESCOM"], return_tensors="pt")
args = (e["input_ids"], e["attention_mask"], e["token_type_ids"])
kw = dict(input_names=["input_ids", "attention_mask", "token_type_ids"], output_names=["logits"], opset_version=17,
          dynamic_axes={k: {0: "batch", 1: "seq"} for k in ["input_ids", "attention_mask", "token_type_ids"]} | {"logits": {0: "batch"}})
try:
    torch.onnx.export(Wrap(model), args, "model_fp32.onnx", dynamo=False, **kw)
except TypeError:
    torch.onnx.export(Wrap(model), args, "model_fp32.onnx", **kw)
quantize_dynamic("model_fp32.onnx", f"{OUT}/model_int8.onnx", weight_type=QuantType.QInt8)
tok.save_pretrained(OUT)
for f in os.listdir(OUT):
    if f not in ("tokenizer.json", "model_int8.onnx"): os.remove(f"{OUT}/{f}")
json.dump(LABELS, open(f"{OUT}/labels.json", "w"))

# Re-test the EXPORTED int8 model through exactly the code path the API uses (tokenizers + onnxruntime, no torch, no transformers).
tk = Tokenizer.from_file(f"{OUT}/tokenizer.json"); tk.enable_truncation(MAXLEN); tk.enable_padding()
sess = ort.InferenceSession(f"{OUT}/model_int8.onnx", providers=["CPUExecutionProvider"])
def onnx_predict(texts):
    out = []
    for i in range(0, len(texts), 128):
        enc = tk.encode_batch(texts[i:i + 128])
        feed = {"input_ids": np.array([x.ids for x in enc], dtype=np.int64), "attention_mask": np.array([x.attention_mask for x in enc], dtype=np.int64),
                "token_type_ids": np.array([x.type_ids for x in enc], dtype=np.int64)}
        out.append(sess.run(None, feed)[0])
    return np.concatenate(out)
logits = onnx_predict([r["text"] for r in test])
int8_pred = [LABELS[i] for i in logits.argmax(-1)]
int8_acc = float(np.mean([p == g for p, g in zip(int8_pred, gold)]))
agree = float(np.mean([a == b for a, b in zip(int8_pred, pred)]))
p = np.exp(logits - logits.max(-1, keepdims=True)); p /= p.sum(-1, keepdims=True); conf = p.max(-1)
rows = []
for th in (0.5, 0.6, 0.7, 0.8, 0.9):
    keep = conf >= th
    rows.append({"threshold": th, "answered": float(keep.mean()), "accuracy_when_answered": float(np.mean([int8_pred[i] == gold[i] for i in np.where(keep)[0]])) if keep.any() else None})
metrics = {"base_model": BASE, "pretrained_weights_loaded": PRETRAINED, "labels": LABELS, "epochs": EPOCHS, "train_rows": len(train), "test_rows": len(test), "test_is": "synthetic, phrases never seen in training",
           "fp32_test_accuracy": fp32_acc, "int8_test_accuracy": int8_acc, "int8_agrees_with_fp32": agree, "macro_f1_fp32": float(f1_score(gold, pred, average="macro")),
           "confidence_table": rows, "model_bytes": os.path.getsize(f"{OUT}/model_int8.onnx")}
json.dump(metrics, open(f"{OUT}/metrics.json", "w"), indent=1)
print(json.dumps({k: v for k, v in metrics.items() if k != "labels"}, indent=1))
with zipfile.ZipFile("taxsetu_narration_model.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUT): z.write(f"{OUT}/{f}", f)
print("zip bytes:", os.path.getsize("taxsetu_narration_model.zip"))

In [ ]:
from google.colab import files
files.download("taxsetu_narration_model.zip")